In [0]:
%pip install geopandas rasterio pandas numpy
%pip install shapely  
%pip install arcgis
%pip install shapely fiona
%pip install pyshp==2.1.3

In [0]:
%restart_python

In [0]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import pandas as pd
import numpy as np
from rasterstats import zonal_stats

from datetime import datetime
import os
import shutil
from arcgis.gis import GIS


In [0]:
# USER INPUT PARAMETERS

# Input Raster
lcm_data = '/Volumes/prd_dash_bronze/uk_ceh_environmental_info_data_centre_unrestricted/land_cover_map_2023_10m/format_GEOTIFF_land_cover_map_2023_10m/LATEST_land_cover_map_2023_10m/gblcm2023_10m.tif'

# Input Vector
sssi_data = '/Volumes/prd_dash_bronze/defra_data_services_platform_unrestricted/sites_of_special_scientific_interest/format_GEOPARQUET_sites_of_special_scientific_interest/LATEST_sites_of_special_scientific_interest/Sites_of_Special_Scientific_Interest_England.parquet'

# AGOL Login details
username = dbutils.secrets.get(scope='arcgis_scope', key='username')
password = dbutils.secrets.get(scope='arcgis_scope', key='password')


In [0]:
# Read in our raster dataset
src = rasterio.open(lcm_data)

# Read in the SSSI layer
sssi_areas = gpd.read_parquet(sssi_data)

lcm_lookup_path = '/Volumes/prd_dash_lab/dash_training_unrestricted/training/ArcGIS/LCM_Class_Table.csv'

In [0]:
def calculate_landcover_stats(
    gdf,
    raster_src,
    name_field="name"
):
    """
    Calculate land cover area per class for each polygon in a GeoDataFrame.

    This function clips a raster dataset to each polygon, counts the number
    of pixels belonging to each land cover class, and converts those counts
    into area using the raster resolution. The output is returned as a wide
    table with one row per polygon and one column per land cover class.

    Parameters
    ----------
    gdf : GeoDataFrame
        Input polygon dataset (e.g. SSSI boundaries). Must be in the same CRS
        as the raster or already reprojected beforehand.
    raster_src : rasterio DatasetReader
        Open raster dataset containing land cover classes in band 1.
    name_field : str, optional
        Column in the GeoDataFrame used as the polygon identifier.
        Default is "name".

    Returns
    -------
    DataFrame
        A pandas DataFrame with one row per polygon and columns for each
        land cover class (1–21), containing area values in square metres.
    """

    results = []

    # Calculate pixel area from raster resolution
    pixel_area = raster_src.res[0] * raster_src.res[1]
    nodata = raster_src.nodata

    # Iterate over each polygon feature
    for idx, row in gdf.iterrows():
        geom = [row.geometry]

        try:
            # Clip raster to polygon extent
            out_image, _ = mask(raster_src, geom, crop=True)
            data = out_image[0]

            # Remove nodata values if defined
            if nodata is not None:
                data = data[data != nodata]

            # Skip polygons with no valid raster data
            if data.size == 0:
                continue

            # Get unique class values and their pixel counts
            classes, counts = np.unique(data, return_counts=True)

            # Convert pixel counts to area
            areas = counts * pixel_area

            # Store results for each class
            for c, count, area in zip(classes, counts, areas):
                results.append({
                    "sssi_name": row[name_field],
                    "land_class": int(c),
                    "area_m2": float(area)
                })

        except Exception as e:
            # Skip problematic geometries
            print(f"Skipping feature {idx}: {e}")

    # Convert results list to DataFrame
    df = pd.DataFrame(results)

    # Remove artificial class 0 values introduced during masking
    df = df[df["land_class"] != 0]

    # Pivot to wide format (one column per land class)
    pivot = df.pivot_table(
        index="sssi_name",
        columns="land_class",
        values="area_m2",
        fill_value=0
    )

    # Ensure all expected classes (1–21) are present as columns
    all_classes = list(range(1, 22))
    pivot = pivot.reindex(columns=all_classes, fill_value=0)

    # Flatten column index and reset index for final output
    pivot.columns.name = None
    pivot = pivot.reset_index()

    return pivot


In [0]:
# HELPER FUNCTIONS

def load_lcm_lookup(csv_path):
    """Load land cover class lookup dictionary"""
    df = pd.read_csv(csv_path)
    return dict(zip(df["LC_Identifier"], df["LC_Class"]))


def apply_class_labels(df, lc_dict):
    """Rename class columns using lookup"""
    return df.rename(columns=lc_dict)


def calculate_percentages(df, id_col="sssi_name"):
    """Convert area table to percentage cover"""
    values = df.set_index(id_col)
    prop = values.div(values.sum(axis=1), axis=0)
    return (prop * 100).reset_index()


def clean_column_names(df):
    """Standardise column names (no spaces etc.)"""
    df.columns = [col.replace(" ", "_") for col in df.columns]
    return df


def join_to_geodataframe(gdf, df, gdf_key="name", df_key="sssi_name"):
    """Join results back to GeoDataFrame"""
    out = gdf.merge(df, left_on=gdf_key, right_on=df_key, how="left")

    # Fill missing values
    out = out.fillna(0)

    # Ensure GeoDataFrame
    return gpd.GeoDataFrame(out, geometry="geometry")

In [0]:
# Load data
sssi_areas = gpd.read_parquet(sssi_data)
sssi_areas = sssi_areas.to_crs(epsg=27700)

# Run your existing function
lcm_area = calculate_landcover_stats(sssi_areas, src)

# Rename class columns
lc_dict = load_lcm_lookup(lcm_lookup_path)
lcm_named = apply_class_labels(lcm_area, lc_dict)

# Convert to %
lcm_percent = calculate_percentages(lcm_named)

# Join back to spatial data
sssi_lcm = join_to_geodataframe(sssi_areas, lcm_percent)

# Clean column names
sssi_lcm = clean_column_names(sssi_lcm)

# Drop unwanted fields
cols_to_drop = [c for c in ['hyperlink', 'contact_no', 'sssi_name'] if c in sssi_lcm.columns]
sssi_lcm = sssi_lcm.drop(columns=cols_to_drop)

# Final result
# display(sssi_lcm)

In [0]:
# PARAMETERS
SIMPLIFY_TOLERANCE = 0.01  # set to None to skip simplification

# Versioning
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
OUTPUT_NAME = f"sssi_landcover_{timestamp}"
AGOL_TITLE = f"SSSI Landcover {timestamp}"

# Local paths
LOCAL_FOLDER = f"/local_disk0/{OUTPUT_NAME}_folder"
zip_path = f"/local_disk0/{OUTPUT_NAME}.zip"

# PREPARE DATA
if SIMPLIFY_TOLERANCE is not None:
    sssi_lcm["geometry"] = sssi_lcm["geometry"].simplify(
        SIMPLIFY_TOLERANCE,
        preserve_topology=True
    )

os.makedirs(LOCAL_FOLDER, exist_ok=True)

# WRITE SHAPEFILE
shp_path = os.path.join(LOCAL_FOLDER, f"{OUTPUT_NAME}.shp")
sssi_lcm.to_file(shp_path)

# ZIP SHAPEFILE
shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir=LOCAL_FOLDER
)

# CONNECT TO AGOL
gis = GIS(
    url="https://defra.maps.arcgis.com/",
    username=username,
    password=password
)

print("Connected to:", gis.properties.name)


# UPLOAD + PUBLISH
item = gis.content.add(
    {
        "title": AGOL_TITLE,
        "type": "Shapefile",
        "tags": f"SSSI, landcover, version_{timestamp}",
        "description": "SSSI land cover percentages from LCM 2023"
    },
    data=zip_path
)

published = item.publish()

print("Feature layer created:")
print(published.url)

In [0]:
# CLEAN UP LOCAL FILES
shutil.rmtree(LOCAL_FOLDER, ignore_errors=True)

if os.path.exists(zip_path):
    os.remove(zip_path)
